In [1]:
!pip install ortools


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import requests
import pandas as pd
import numpy as np
from ortools.linear_solver import pywraplp

In [3]:
DATA_DIR = Path("data/")
DATA_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR

WindowsPath('data')

In [4]:
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"


def download_taxi_data(year, month, taxi_type="yellow"):

    filename = f"{taxi_type}_tripdata_{year}-{month:02d}.parquet"

    url = f"{BASE_URL}/{filename}"

    output_path = DATA_DIR / filename

    print(f"Descargando {filename}...")

    response = requests.get(url, stream=True)
    response.raise_for_status()

    with open(output_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    print(f"Guardado en: {output_path}")

    return output_path

In [5]:
file_jan = download_taxi_data(2025, 1)
file_feb = download_taxi_data(2025, 2)

Descargando yellow_tripdata_2025-01.parquet...
Guardado en: data\yellow_tripdata_2025-01.parquet
Descargando yellow_tripdata_2025-02.parquet...
Guardado en: data\yellow_tripdata_2025-02.parquet


In [6]:
df_jan = pd.read_parquet(file_jan)
df_feb = pd.read_parquet(file_feb)

print("Enero:", df_jan.shape)
print("Febrero:", df_feb.shape)

Enero: (3475226, 20)
Febrero: (3577543, 20)


In [7]:
df = pd.concat([df_jan, df_feb], ignore_index=True)

df.to_parquet(DATA_DIR / "yellow_tripdata.parquet", index=False)

print(df.shape)

(7052769, 20)


In [8]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [9]:
cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "passenger_count",
    "fare_amount",
    "total_amount",
]

df = df[cols]

In [10]:
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])

df["hour"] = df["tpep_pickup_datetime"].dt.hour
df["day"] = df["tpep_pickup_datetime"].dt.day_name()

In [11]:
zone_hour_demand = df.groupby(["day", "hour"]).size().reset_index(name="total_requests")

peak_demand = zone_hour_demand.sort_values(
    by="total_requests", ascending=False
).reset_index(drop=True)

daily_peak = peak_demand.groupby("day").first().reset_index()

zone_requests = (
    df.groupby(["day", "hour", "PULocationID"]).size().reset_index(name="requests")
)

peak_zone_requests = zone_requests.merge(
    daily_peak[["day", "hour", "total_requests"]], on=["day", "hour"], how="inner"
)

scenario_matrix = (
    peak_zone_requests.pivot_table(
        index=["day", "hour", "total_requests"],
        columns="PULocationID",
        values="requests",
        fill_value=0,
    )
    .reset_index()
    .rename(columns={"hour": "peak_hour"})
)

scenario_matrix.columns.name = None

day_order = {
    "Monday": 1,
    "Tuesday": 2,
    "Wednesday": 3,
    "Thursday": 4,
    "Friday": 5,
    "Saturday": 6,
    "Sunday": 7,
}

scenario_matrix["day_order"] = scenario_matrix["day"].map(day_order)

scenario_matrix = (
    scenario_matrix.sort_values("day_order")
    .drop(columns="day_order")
    .reset_index(drop=True)
)

scenario_matrix

,day,peak_hour,total_requests,1,3,4,6,7,8,9,...,256,257,258,259,260,261,262,263,264,265
0,Monday,18,57379,6.0,0.0,37.0,1.0,53.0,1.0,0.0,...,35.0,2.0,6.0,1.0,30.0,321.0,717.0,970.0,117.0,12.0
1,Tuesday,18,76484,3.0,3.0,43.0,0.0,38.0,0.0,0.0,...,31.0,2.0,3.0,0.0,18.0,520.0,916.0,1154.0,153.0,19.0
2,Wednesday,18,86198,8.0,1.0,84.0,0.0,41.0,0.0,0.0,...,36.0,2.0,5.0,1.0,22.0,602.0,1029.0,1314.0,193.0,17.0
3,Thursday,18,98576,3.0,1.0,140.0,1.0,34.0,2.0,0.0,...,54.0,2.0,2.0,0.0,21.0,607.0,1201.0,1462.0,206.0,14.0
4,Friday,18,96567,5.0,1.0,236.0,0.0,114.0,0.0,3.0,...,73.0,8.0,8.0,5.0,33.0,558.0,1383.0,2026.0,186.0,13.0
5,Saturday,19,73870,4.0,0.0,282.0,0.0,75.0,0.0,0.0,...,106.0,10.0,5.0,5.0,50.0,393.0,898.0,1615.0,118.0,10.0
6,Sunday,0,62624,0.0,0.0,680.0,0.0,64.0,0.0,2.0,...,198.0,5.0,14.0,4.0,20.0,194.0,155.0,910.0,99.0,21.0


In [12]:
taxi_supply_matrices = {}

zone_columns = scenario_matrix.columns[3:]

for _, row in scenario_matrix.iterrows():

    day = row["day"]

    demand = row[zone_columns].astype(int)

    multipliers = np.random.uniform(0.5, 1.3, size=len(zone_columns))

    initial_supply = (demand.values * multipliers).astype(int)

    total_demand = demand.sum()

    target_total_supply = int(total_demand * 0.85)

    scaling_factor = target_total_supply / initial_supply.sum()

    final_supply = (initial_supply * scaling_factor).astype(int)

    supply_df = pd.DataFrame(
        {
            "zone": zone_columns.astype(int),
            "demand": demand.values,
            "available_taxis": final_supply,
        }
    )

    taxi_supply_matrices[day] = supply_df

In [13]:
taxi_supply_matrices["Monday"].head(5)

,zone,demand,available_taxis
0,1,6,4
1,3,0,0
2,4,37,32
3,6,1,0
4,7,53,47


In [14]:
zones = sorted(zone_columns.astype(int))

cost_matrix = pd.DataFrame(index=zones, columns=zones)

zone_type = {
    z: np.random.choice(["central", "normal", "peripheral"])
    for z in zones
}

for i in zones:
    for j in zones:

        if i == j:
            cost = 0

        else:

            base = np.random.rand()

            if zone_type[i] == "central" and zone_type[j] == "central":
                cost = np.random.randint(1, 5)

            elif zone_type[i] == "peripheral" or zone_type[j] == "peripheral":
                cost = np.random.randint(10, 31)

            else:
                cost = np.random.randint(5, 20)

            cost += np.random.randint(0, 3)

        cost_matrix.loc[i, j] = cost

cost_matrix = cost_matrix.astype(int)
cost_matrix

,1,3,4,6,7,8,9,10,11,12,...,256,257,258,259,260,261,262,263,264,265
1,0,10,4,3,15,6,30,31,27,25,...,17,5,1,3,5,22,27,14,2,8
3,11,0,16,20,21,19,10,19,22,13,...,16,12,16,13,6,17,27,22,16,21
4,2,14,0,5,24,3,19,15,11,30,...,20,4,1,3,4,27,22,20,6,10
6,4,14,4,0,20,3,31,15,23,11,...,20,3,4,6,4,26,20,13,2,6
7,17,23,27,21,0,15,20,14,19,25,...,19,26,12,17,11,30,18,27,10,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,18,27,22,25,31,19,17,21,12,11,...,16,21,15,16,19,0,17,18,23,15
262,15,18,30,14,13,11,10,22,22,29,...,10,27,26,25,22,26,0,25,14,11
263,20,18,29,21,17,26,16,21,29,25,...,24,12,14,19,14,14,30,0,17,29
264,5,15,4,3,24,5,21,29,27,30,...,18,3,2,3,5,29,15,24,0,9


In [15]:
critical_zones = [132, 138, 161, 230, 236]

MIN_CRITICAL_COVERAGE = 0.95

MAX_MOVEMENT_COST = 20

MIN_LOCAL_REMAINING = 0.30


def solve_taxi_redistribution(day):

    supply_df = taxi_supply_matrices[day].copy()

    zones = supply_df["zone"].tolist()

    demand = dict(zip(supply_df["zone"], supply_df["demand"]))

    supply = dict(zip(supply_df["zone"], supply_df["available_taxis"]))

    solver = pywraplp.Solver.CreateSolver("SCIP")

    x = {}

    for i in zones:
        for j in zones:

            # bloquear movimientos absurdos
            if cost_matrix.loc[i, j] > MAX_MOVEMENT_COST:

                x[i, j] = None

            else:

                x[i, j] = solver.IntVar(0, solver.infinity(), f"x_{i}_{j}")

    uncovered = {}

    for j in zones:

        uncovered[j] = solver.NumVar(0, solver.infinity(), f"uncovered_{j}")

    for i in zones:

        outgoing = []

        for j in zones:

            if x[i, j] is not None:

                outgoing.append(x[i, j])

        solver.Add(sum(outgoing) <= supply[i] * (1 - MIN_LOCAL_REMAINING))

    for j in zones:

        incoming = []

        outgoing = []

        for i in zones:

            if x[i, j] is not None:

                incoming.append(x[i, j])

        for k in zones:

            if x[j, k] is not None:

                outgoing.append(x[j, k])

        solver.Add(
            (supply[j] - sum(outgoing) + sum(incoming) + uncovered[j]) >= demand[j]
        )

    for z in critical_zones:

        if z in demand:

            solver.Add(uncovered[z] <= demand[z] * (1 - MIN_CRITICAL_COVERAGE))

    movement_cost = solver.Sum(
        cost_matrix.loc[i, j] * x[i, j]
        for i in zones
        for j in zones
        if x[i, j] is not None
    )

    uncovered_penalty = solver.Sum(uncovered[j] * 1000 for j in zones)

    solver.Minimize(movement_cost + uncovered_penalty)

    status = solver.Solve()

    movements = []

    if status == pywraplp.Solver.OPTIMAL:

        for i in zones:
            for j in zones:

                if x[i, j] is not None:

                    value = int(x[i, j].solution_value())

                    if value > 0:

                        movements.append(
                            {
                                "from_zone": i,
                                "to_zone": j,
                                "taxis_moved": value,
                                "cost": cost_matrix.loc[i, j],
                            }
                        )

    movement_df = pd.DataFrame(movements)

    uncovered_results = pd.DataFrame(
        {
            "zone": zones,
            "uncovered_requests": [uncovered[z].solution_value() for z in zones],
        }
    )

    return {"status": status, "movements": movement_df, "uncovered": uncovered_results}

In [16]:
day = "Monday"

result = solve_taxi_redistribution(day)

movements = result["movements"]
uncovered = result["uncovered"]

total_requests = taxi_supply_matrices[day]["demand"].sum()

total_uncovered = int(uncovered["uncovered_requests"].sum())

total_attended = total_requests - total_uncovered

total_cost = int((movements["taxis_moved"] * movements["cost"]).sum())

print("-" * 70)
print(f" RESULTADOS OPTIMIZACIÓN - {day.upper()} ".center(70, "-"))
print("-" * 70)

print(f"\nTotal requests: {total_requests:,}")
print(f"Máx requests atendidos: {total_attended:,}")
print(f"Requests no cubiertos: {total_uncovered:,}")
print(f"Costo mínimo total: {total_cost:,}")

print("\n" + "-" * 70)
print(" TOP MOVIMIENTOS DE TAXIS ".center(70, "-"))
print("-" * 70 + "\n")

top_moves = movements.sort_values("taxis_moved", ascending=False).head(10)

for _, row in top_moves.iterrows():

    print(
        f"Zona {row['from_zone']} "
        f"→ Zona {row['to_zone']} "
        f"| taxis movidos: {int(row['taxis_moved'])} "
        f"| costo: {row['cost']}"
    )

print("\n" + "-" * 70)
print(" ZONAS CON MÁS REQUESTS NO CUBIERTOS ".center(70, "-"))
print("-" * 70 + "\n")

top_uncovered = uncovered.sort_values("uncovered_requests", ascending=False).head(10)

for _, row in top_uncovered.iterrows():

    print(
        f"Zona {row['zone']} " f"→ no cubiertos: " f"{int(row['uncovered_requests'])}"
    )

----------------------------------------------------------------------
------------------ RESULTADOS OPTIMIZACIÓN - MONDAY ------------------
----------------------------------------------------------------------

Total requests: 57,379
Máx requests atendidos: 48,662
Requests no cubiertos: 8,717
Costo mínimo total: 10,130

----------------------------------------------------------------------
---------------------- TOP MOVIMIENTOS DE TAXIS ----------------------
----------------------------------------------------------------------

Zona 186 → Zona 162 | taxis movidos: 292 | costo: 10
Zona 233 → Zona 230 | taxis movidos: 226 | costo: 10
Zona 114 → Zona 138 | taxis movidos: 119 | costo: 6
Zona 239 → Zona 142 | taxis movidos: 89 | costo: 2
Zona 236 → Zona 143 | taxis movidos: 79 | costo: 1
Zona 90 → Zona 230 | taxis movidos: 49 | costo: 16
Zona 166 → Zona 138 | taxis movidos: 47 | costo: 12
Zona 141 → Zona 132 | taxis movidos: 42 | costo: 6
Zona 236 → Zona 138 | taxis movidos: 40 | costo

In [17]:
def check_model_constraints(day, result, taxi_supply_matrices, critical_zones):
    movements = result["movements"]
    uncovered = result["uncovered"]
    supply_df = taxi_supply_matrices[day].set_index("zone")
    
    print("-" * 70)
    print(f" AUDITORIA DE RESTRICCIONES - {day.upper()} ".center(70, "-"))
    print("-" * 70)

    print("\nRESTRICCION 1: COBERTURA EN ZONAS CRITICAS \n")
    print(f"{'ZONA':<10} | {'DEMANDA':<10} | {'NO CUBIERTO':<12} | {'COBERTURA':<12} | {'ESTADO'}")
    print("-" * 70)
    
    crit_met = True
    for z in critical_zones:
        if z in supply_df.index:
            demand_z = supply_df.loc[z, "demand"]
            uncovered_z = uncovered.set_index("zone").loc[z, "uncovered_requests"]
            
            coverage = 1.0 if demand_z == 0 else 1 - (uncovered_z / demand_z)
            
            passed = coverage >= (MIN_CRITICAL_COVERAGE - 1e-4)
            status = "CUMPLE" if passed else "VIOLADO"
            if not passed: crit_met = False
            
            print(f"{z:<10} | {int(demand_z):<10} | {int(uncovered_z):<12} | {coverage*100:10.2f}% | {status}")
    
    print(f"\n>> RESULTADO GLOBAL CRITICAS: {'OK' if crit_met else 'ERROR'}")

    print("\n" + "-" * 70)
    print("\nRESTRICCION 2: LIMITE DE SALIDA (RETENCION LOCAL) \n")
    print(f"{'ZONA':<10} | {'OFERTA':<10} | {'MAX SALIDA':<12} | {'SALIDA REAL':<12} | {'ESTADO'}")
    print("-" * 70)
    
    outgoing_totals = movements.groupby("from_zone")["taxis_moved"].sum()
    retention_met = True
    
    for z in supply_df.index:
        supply_z = supply_df.loc[z, "available_taxis"]
        max_out = supply_z * (1 - MIN_LOCAL_REMAINING)
        actual_out = outgoing_totals.get(z, 0)
        
        passed = actual_out <= (max_out + 1e-4)
        status = "CUMPLE" if passed else "VIOLADO"
        if not passed: retention_met = False
        
        if actual_out > 0:
            print(f"{z:<10} | {int(supply_z):<10} | {int(max_out):<12} | {int(actual_out):<12} | {status}")
            
    print(f"\n>> RESULTADO GLOBAL RETENCION: {'OK' if retention_met else 'ERROR'}")

    print("\n" + "-" * 70)
    print("\n[ RESTRICCION 3: DISTANCIA / COSTO MAXIMO ]")
    
    illegal_moves = movements[movements["cost"] > MAX_MOVEMENT_COST]
    if illegal_moves.empty:
        print(f"\nOK: No hay movimientos que superen el costo maximo de {MAX_MOVEMENT_COST}")
    else:
        print(f"ERROR: Se detectaron {len(illegal_moves)} movimientos prohibidos")

check_model_constraints(day, result, taxi_supply_matrices, critical_zones)

----------------------------------------------------------------------
---------------- AUDITORIA DE RESTRICCIONES - MONDAY -----------------
----------------------------------------------------------------------

RESTRICCION 1: COBERTURA EN ZONAS CRITICAS 

ZONA       | DEMANDA    | NO CUBIERTO  | COBERTURA    | ESTADO
----------------------------------------------------------------------
132        | 2471       | 77           |      96.88% | CUMPLE
138        | 1519       | 75           |      95.06% | CUMPLE
161        | 4126       | 172          |      95.83% | CUMPLE
230        | 1955       | 97           |      95.04% | CUMPLE
236        | 2565       | 0            |     100.00% | CUMPLE

>> RESULTADO GLOBAL CRITICAS: OK

----------------------------------------------------------------------

RESTRICCION 2: LIMITE DE SALIDA (RETENCION LOCAL) 

ZONA       | OFERTA     | MAX SALIDA   | SALIDA REAL  | ESTADO
----------------------------------------------------------------------
1   